In [1]:
from llama_index.llms.groq import Groq
from llama_index.core import SimpleDirectoryReader, StorageContext, load_index_from_storage
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
#from sentence_transformers import SentenceTransformer
from llama_index.core import VectorStoreIndex, Settings
from llama_index.core import PromptTemplate


import os

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
#legacy keras weil fehler mit huggingface und dem aktuellen sentence transformer
#pip install tf-keras

## setting up model

In [2]:
# This info's at the top of each HuggingFace model page
model = "llama-3.3-70b-versatile"

llm = Groq(
    model=model,
     api_key=os.environ.get(
         "GROQ_API_KEY"
     ),  # you can also enter your API key here, either hard-coded or read from another file
)

## load data

In [3]:
documents = SimpleDirectoryReader(r"D:\Mein Zeug\Arbeit\Data_Science_Bootcamp\anaconda vs enivornment\hello_ds\ML supervised\AI\cheap_buddha2\data").load_data()

## split sentences in documents

In [4]:
text_splitter = SentenceSplitter(chunk_size=800, chunk_overlap=150)

docs = text_splitter.get_nodes_from_documents(documents)

In [5]:
#how many docs?
len(docs)

7332

## create embeddings

In [6]:
# embeddings

embedding_model = "sentence-transformers/all-MiniLM-L6-v2"  # or 4B if your machine can handle it
embeddings_folder = r"D:\Mein Zeug\Arbeit\Data_Science_Bootcamp\anaconda vs enivornment\hello_ds\ML supervised\AI\cheap_buddha2\embedding_model"


embeddings = HuggingFaceEmbedding(
    model_name=embedding_model, cache_folder=embeddings_folder
)

2026-03-01 16:46:48,565 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\nicot\anaconda3\envs\myenv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\Mein Zeug\Arbeit\Data_Science_Bootcamp\anaconda vs enivornment\hello_ds\ML supervised\AI\cheap_buddha2\embedding_model\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## creating vector database

In [ ]:
#Fix the version mismatch (do this first)
#pip install -U "sentence-transformers>=3.1.0"

: 

In [7]:
#create a vector index
vector_index = VectorStoreIndex.from_documents(
    documents, transformations=[text_splitter], embed_model=embeddings
)

In [8]:
#save the vector index
vector_index.storage_context.persist(persist_dir=r"D:\Mein Zeug\Arbeit\Data_Science_Bootcamp\anaconda vs enivornment\hello_ds\ML supervised\AI\cheap_buddha2\vector_index")

In [ ]:
#retrieve saved vector database

#storage_context = StorageContext.from_defaults(persist_dir=r"D:\Mein Zeug\Arbeit\Data_Science_Bootcamp\anaconda vs enivornment\hello_ds\ML supervised\AI\buddhism_chatbot\vector_index")
#vector_index = load_index_from_storage(storage_context, embed_model=embeddings)

In [9]:
# monk prompt template

monk_qa_tmpl = PromptTemplate(
    """Context information is below.
---------------------
{context_str}
---------------------

Answer the question using only the context.
Speak like a Tibetan person with a university degree (speak like a young professor from Tibet).

Query: {query_str}
Answer:"""
)

monk_refine_tmpl = PromptTemplate(
    """The original query is: {query_str}

We already have an answer:
{existing_answer}

We have more context below:
------------
{context_msg}
------------

Refine the answer if needed, using only the new context.
Keep the professor speaking style.
If the new context is not useful, return the original answer unchanged.

Refined Answer:"""
)

In [10]:
#monk engine engine
# load in the documents

# query engine
query_engine2 = vector_index.as_query_engine(
    llm=llm,
    text_qa_template=monk_qa_tmpl,
    similarity_top_k=2,
    response_mode="compact",
    refine_template=monk_refine_tmpl,
)

In [11]:
answer = query_engine2.query("Who is Buddha exactly?")

print(answer)

2026-03-01 16:51:17,242 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Tashi delek, my friend. As a Tibetan scholar, I must clarify that the concept of Buddha is multifaceted and profound. According to our scriptures, a Buddha is "one who has woken up" from the mental sleep of the untrained mind, or "one who knows" the Dharma, the basic truth of things. In essence, a Buddha is an individual who has achieved the goal of the Buddhist path, having fully awakened to the true nature of reality.

However, when we refer to "the Buddha," we are often talking about the historical figure, Shakyamuni, who lived in India over 2,500 years ago and founded our beloved tradition. But, as our texts reveal, Shakyamuni is not the only Buddha; he is one of many Buddhas who have appeared in the universe, each with their own unique role and significance.

In Mahayana Buddhism, we understand that Buddhahood is not limited to a single individual, but rather it is a state of enlightenment that can be achieved by anyone who strives for perfection and removes their ignorance throug

getting this on streamlit ... create .py file

(myenv) C:\Users\nicot>streamlit run D:\Mein Zeug\Arbeit\Data_Science_Bootcamp\anaconda vs enivornment\hello_ds\ML supervised\AI\5.5_streamlit.py
Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Streamlit requires raw Python (.py) files, but the provided file has no extension.
For more information, please see https://docs.streamlit.io

(myenv) C:\Users\nicot>python -m streamlit run "D:\Mein Zeug\Arbeit\Data_Science_Bootcamp\anaconda vs enivornment\hello_ds\ML supervised\AI\5.5_streamlit.py"

      Welcome to Streamlit!

      If you'd like to receive helpful onboarding emails, news, offers, promotions,
      and the occasional swag, please enter your email address below. Otherwise,
      leave this field blank.

      Email:

  You can find our privacy policy at https://streamlit.io/privacy-policy

  Summary:
  - This open source library collects usage statistics.
  - We cannot see and do not store information contained inside Streamlit apps,
    such as text, charts, images, etc.
  - Telemetry data is stored in servers in the United States.
  - If you'd like to opt out, add the following to %userprofile%/.streamlit/config.toml,
    creating that file if necessary:

    [browser]
    gatherUsageStats = false


  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501